In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
from datetime import datetime

GESTURES = [
    "jump", "crouch", "move_left", "move_right",
    "quick_punch", "kick",
    "attack"
]

FRAMES_PER_CLIP = 30   # 1 second at ~30fps
CLIPS_PER_GESTURE = 3

OUTPUT_DIR = "karate_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for g in GESTURES:
    os.makedirs(os.path.join(OUTPUT_DIR, g), exist_ok=True)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

cap = cv2.VideoCapture(0)

def capture_clip(label, clip_no):
    keypoints_list = []
    print(f"\nPrepare for {label} clip {clip_no+1}")
    for i in range(3, 0, -1):
        print(f"Starting in {i}...")
        cv2.waitKey(1000)

    print("Recording...")
    frame_count = 0
    while frame_count < FRAMES_PER_CLIP:
        ret, frame = cap.read()
        if not ret:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = pose.process(rgb)

        if result.pose_landmarks:
            kps = [[lm.x, lm.y, lm.z] for lm in result.pose_landmarks.landmark]
            keypoints_list.append(kps)
        else:
            keypoints_list.append(np.zeros((33,3)))

        cv2.putText(frame, f"{label} clip {clip_no+1}", (10,50),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
        cv2.imshow("Recording", frame)
        frame_count += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    np.save(os.path.join(OUTPUT_DIR, label, f"{timestamp}.npy"), np.array(keypoints_list))
    print(f"Saved {label} clip {clip_no+1}")

# Collect all gestures
for g in GESTURES:
    for clip_no in range(CLIPS_PER_GESTURE):
        capture_clip(g, clip_no)

cap.release()
cv2.destroyAllWindows()
print("Data collection complete.")
